# AREA: fine-tuning Laya-multilingual on the MIT causal taxonomy

Fine-tunes [`convaiinnovations/laya-multilingual`](https://huggingface.co/convaiinnovations/laya-multilingual)
on the causal labels (entity, intent, timing) of the MIT AI Risk Repository with the scripts of the
[AREA repository](https://github.com/6MicheleStingo9/AREA/tree/rework/area) (`training/`).

Three variants are trained on the MIT training split without its validation groups and calibrated on
those groups; the one with the best macro-F1 on validation is then evaluated, once, on the MIT eval
split and on the AREA risks.

**Settings:** Accelerator *GPU T4 x2* (or P100), Internet *on*; then *Save Version → Save & Run All*
(about an hour).

**Output** (`/kaggle/working/`): `laya-causal-v2.zip` (the chosen checkpoint, about 0.6 GB) and
`results/` (test-set metrics, predictions and the validation comparison in `selection.json`).

In [ ]:
%env USE_TF=0
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 -b rework/area https://github.com/6MicheleStingo9/AREA.git
%cd AREA
!git log -1 --format='%h %s'
!pip install -q laya==0.4.1

Training, validation and test files in Laya's format (`training/data/`): one row per risk, three typed questions.

In [ ]:
!python -m training.build_laya_data

Three variants: the first run's settings (4 epochs), 8 epochs, and 8 epochs with the rarest classes oversampled.

In [ ]:
!python -m training.finetune_laya --name v4 --epochs 4
!python -m training.finetune_laya --name v8 --epochs 8
!python -m training.finetune_laya --name v8os --epochs 8 --oversample

Choice on the validation split: highest macro-F1 averaged over the three axes.

In [ ]:
!python -m training.select_laya v4 v8 v8os
import json
best = json.load(open("training/checkpoints/selection.json"))["best"]
print("chosen:", best)

The chosen variant on the two test sets (used only here).

In [ ]:
!cp -r training/checkpoints/{best} training/checkpoints/laya-causal-v2
!python -m training.evaluate_laya --model training/checkpoints/laya-causal-v2 --name laya-causal-v2

Package the checkpoint and the results for download, then drop the clone so *Download All* stays small.

In [ ]:
!mkdir -p /kaggle/working/results && cp evaluation/results/*laya-causal-v2* training/checkpoints/selection.json /kaggle/working/results/
!cd training/checkpoints && zip -q -r /kaggle/working/laya-causal-v2.zip laya-causal-v2
%cd /kaggle/working
!rm -rf AREA
!ls -lh /kaggle/working /kaggle/working/results